# Structural Validator (Goal 0)

## What this notebook does

Before we simulate anything, a rocket design has to be *possible* — structurally valid in the KSP sense.
This notebook builds the validator that enforces that.

## The rocket format

For now, a rocket is a plain Python dict. This keeps things inspectable and printable.
A later notebook (`rocket_representation.ipynb`) will build a class that produces this same format —
so the validator doesn't need to change when that happens.

## What makes a rocket structurally valid?

A valid design must satisfy all of the following:

- **One connected part tree** — all parts form a single connected graph, no orphans
- **Exactly one root part** — one part has no parent (the top of the stack)
- **At least one command source** — a part with `is_command = True`
- **At least one engine** — a part with engine data
- **Propellant compatibility** — every engine has the resources it needs available somewhere in the design
- **Valid part references** — every `type` field refers to a real part in the parts library
- **Valid attachment nodes** — parent/child attachments reference nodes that actually exist on those parts
- **No cycles** — the part tree has no loops
- **Valid stage references** — stage numbers are non-negative integers

We'll build and test one check at a time.

In [9]:
### setup block

from pathlib import Path
import json

parts_json_file = Path('../data/parts_library.json')
with open(parts_json_file) as f:
    parts_library = json.load(f)

parts_by_name = {p['name']: p for  p in parts_library }

rocket_example = {
    "parts": [
        {"id": "pod_0",  "type": "mk1-3pod", "parent": None},
        {"id": "tank_0", "type": "fuelTank", "parent": "pod_0", "attach_node": "bottom"  },
        {"id": "eng_0",  "type": "liquidEngine", "parent": "tank_0", "attach_node": "bottom"},
    ],
    "stages": {"eng_0": 0}
}

for part in rocket_example["parts"]:
    print(part["parent"])


None
pod_0
tank_0


first, load in json parts list, inspect format, and create dict of categories/parts. then a function to check against

In [2]:
def check_part_call(part: str,
                    parts_by_name_dict: dict):
    if part not in parts_by_name_dict:
        return False
    else:
        return True

print(check_part_call('liquidEngine', parts_by_name))
print(check_part_call('nope', parts_by_name))


True
False


check minimal parts for rocket. built individual checks and then final helper

In [3]:
def check_single_root(rocket_dict: dict):
    num_no_parents = 0
    for part in rocket_dict['parts']:
        if part['parent'] is None:
            num_no_parents += 1

    if num_no_parents != 1:
        return False
    return True

def check_has_command(rocket_dict: dict,
                      parts_dict: dict):
    command_count = 0
    for part in rocket_dict['parts']:
        part_name = part['type']
        part_info = parts_dict[part_name]
        is_command = part_info['is_command']
        if is_command:
            command_count += 1
    if command_count > 0:
        return True
    return False

def check_has_engine(rocket_dict: dict,
                     parts_dict: dict):
    engine_count = 0
    for part in rocket_dict['parts']:
        part_name = part['type']
        part_info = parts_dict[part_name]
        is_engine = part_info['engine']
        if is_engine:
            engine_count += 1
    if engine_count > 0:
        return True
    return False


def has_minimal_structure(rocket_dict: dict,
                          parts_dict: dict):
    root_check = check_single_root(rocket_dict)
    command_check = check_has_command(rocket_dict, parts_dict)
    engine_check = check_has_engine(rocket_dict, parts_dict)
    valid_struct = all([root_check, command_check, engine_check])

    return valid_struct


print(check_single_root(rocket_example))
print(check_has_command(rocket_example, parts_by_name))
print(check_has_engine(rocket_example, parts_by_name))
print(has_minimal_structure(rocket_example, parts_by_name))


True
True
True
True


check ship structure

In [4]:
def check_graph_connections(rocket_dict:dict,
                            parts_dict: dict,
                            verbose: bool = False):
    all_parts = {part['id'] for part in rocket_dict['parts']}
    root_id = next(part['id'] for part in rocket_dict['parts'] if part['parent'] is None)
    children = {part['id']: [] for part in rocket_dict['parts']}
    for part in rocket_dict['parts']:
        if part['parent'] is not None:
            children[part['parent']].append(part['id'])

    queue = [root_id]
    visited = set()

    while queue:
        current = queue.pop(0)
        visited.add(current)
        for child in children[current]: ##### Note: this is a check of circularity because KSP enforces a linear graph structure on its rockets. This would not exist on a real rocket that has various circular systems
            if child in visited:
                return False
            else:
                queue.append(child)

    if visited != all_parts:
        return False
    if verbose:
        print(visited)
        return True
    return True

check_graph_connections(rocket_example, parts_by_name)


True

check propellant compatibility


In [5]:

def check_propellant(rocket_dict: dict,
                     parts_dict: dict):
    available_resources = set()
    for part in rocket_dict['parts']:
        part_info = parts_dict[part['type']]
        if part_info['resources']:
            available_resources.update(part_info['resources'].keys())

    needed_propellants = set()
    for part in rocket_dict['parts']:
        if parts_dict[part['type']]['engine'] is not None:
            needed_propellants.update(parts_dict[part['type']]['engine']['propellants'].keys())
    if needed_propellants.issubset(available_resources):
        return True
    return False

check_propellant(rocket_example, parts_by_name)

True

make sure that all parts that are staged are in the rocket dict

In [6]:
def check_staging(rocket_dict: dict):

    all_parts = {part['id'] for part in rocket_dict['parts']}

    stages = set()
    for name, stage  in rocket_dict['stages'].items():
        if not isinstance(stage, int) or stage < 0:
            return False
        stages.add(name)

    if stages.issubset(all_parts):
        return True
    return False

print(check_staging(rocket_example))

True


check node connections

In [7]:
def check_valid_nodes(rocket_dict: dict,
                      parts_dict: dict):
    id_to_type = {p['id']: p['type'] for p in rocket_dict['parts']}


    for part in rocket_dict['parts']:
        if part['parent'] == None:
            continue
        parent_type = id_to_type[part['parent']]
        if not isinstance(part['attach_node'], str):
            return False
        if part['attach_node'] not in parts_dict[parent_type]['nodes'].keys():
            return False
    return True

check_valid_nodes(rocket_example, parts_by_name)

True

make the final wrapper

In [8]:
def validate_rocket(rocket_dict: dict,
                    parts_dict: dict,
                    verbose: bool = False):
    for part in rocket_dict['parts']:
        if not check_part_call(part['type'], parts_dict):
            if verbose:
                print(f"FAIL: unknown part type '{part['type']}")
            return False
    checks = [
          (has_minimal_structure(rocket_dict, parts_dict),  "missing root, command, or engine"),
          (check_graph_connections(rocket_dict, parts_dict), "part tree disconnected or has cycles"),
          (check_propellant(rocket_dict, parts_dict),        "propellant incompatibility"),
          (check_staging(rocket_dict),                       "invalid stage references"),
          (check_valid_nodes(rocket_dict, parts_dict),       "invalid attachment nodes"),
      ]
    for result, message in checks:
          if not result:
              if verbose:
                  print(f"FAIL: {message}")
              return False
    return True

validate_rocket(rocket_example, parts_by_name)

True